# Лабораторная 3. Данные: разметка, дубликаты, разбиение, аугментации

Задание целиком: `labs/lab03/README.md` в репозитории курса.

Ноутбук один на обе части:
- **Часть 1** (после лекции 4): проверка разметки, перевод в COCO и YOLO, дубликаты, распределения;
- **Часть 2** (после лекции 5): группы, честное разбиение, аугментации.

Как работать:
1. В первой ячейке впишите свой номер варианта.
2. Заполняйте ячейки с пометкой `TODO` сверху вниз. Ячейки без `TODO` уже готовы, их менять не нужно.
3. Пока вы не дошли до части 2, её ячейки можно не трогать: Run All проходит через них без ошибок.
4. В конце выполните Run All с нуля, скачайте ноутбук и `results.zip`.

В Kaggle включите интернет: Settings → Internet → On.

## 0. Установка и загрузка данных

Та же ячейка, что в лабах 1 и 2. Ничего в ней менять не нужно, кроме `VARIANT`.

In [ ]:
VARIANT = None   # TODO: ваш порядковый номер в списке группы, например 7
DATA_SOURCE = None  # запасной путь: папка с архивами variant_NN.zip

import os
import sys
import inspect
import urllib.request
from pathlib import Path

assert isinstance(VARIANT, int) and 0 <= VARIANT <= 22, "Впишите свой номер варианта: целое число от 1 до 22"
if VARIANT == 0:
    print("ВНИМАНИЕ: вариант 00 демонстрационный, работа на нём не засчитывается")

if "google.colab" in sys.modules:
    ENV, DATA_ROOT = "colab", Path("/content/data")
elif Path("/kaggle/working").exists():
    ENV, DATA_ROOT = "kaggle", Path("/kaggle/working/data")
elif Path("../../data/get_data.py").exists():
    # Ноутбук открыт прямо из клона репозитория курса: данные кладём в его папку data/
    ENV, DATA_ROOT = "local", Path("../../data").resolve()
else:
    # Локально: ноутбук лежит в cv-2026-familiya/lab03/, данные кладём в cv-2026-familiya/data/.
    # Эта папка указана в .gitignore, в репозиторий она не попадёт.
    ENV, DATA_ROOT = "local", Path("..").resolve() / "data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("среда:", ENV, "| папка данных:", DATA_ROOT)

GET_DATA_URL = "https://raw.githubusercontent.com/ykvnkm/computer-vision-course/main/data/get_data.py"
course_copy = Path("../../data/get_data.py")
kaggle_copies = sorted(Path("/kaggle/input").glob("**/get_data.py")) if ENV == "kaggle" else []
if course_copy.exists():
    script_dir = course_copy.resolve().parent
elif kaggle_copies:
    # Kaggle с подключённым датасетом-зеркалом курса: интернет не нужен
    script_dir = kaggle_copies[0].parent
    print("get_data.py из подключённого датасета:", kaggle_copies[0])
else:
    script_dir = DATA_ROOT
    if not (script_dir / "get_data.py").exists():
        try:
            urllib.request.urlretrieve(GET_DATA_URL, script_dir / "get_data.py")
        except Exception as e:
            raise RuntimeError("Не удалось скачать get_data.py. В Kaggle включите Internet в Settings; "
                               "в остальных случаях проверьте подключение к сети.") from e
sys.path.insert(0, str(script_dir))
from get_data import ensure_variant

params = inspect.signature(ensure_variant).parameters
kwargs = {"dest": DATA_ROOT} if "dest" in params else {}
if DATA_SOURCE and "source" in params:
    kwargs["source"] = DATA_SOURCE
DATA_DIR = Path(ensure_variant(VARIANT, **kwargs))
print("данные варианта:", DATA_DIR)
print(sorted(p.name for p in DATA_DIR.iterdir()))

In [ ]:
import hashlib
import json
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import imagehash
import albumentations as A
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

IMG_DIR = DATA_DIR / "images"
TRIMAP_DIR = DATA_DIR / "annotations" / "trimaps"
XML_DIR = DATA_DIR / "annotations" / "xmls"
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
SEED = 2026

labels = pd.read_csv(DATA_DIR / "labels.csv")
variant_info = json.loads((DATA_DIR / "variant.json").read_text(encoding="utf-8"))
CLASSES = list(variant_info["breeds"])          # индекс в списке = class_id из labels.csv
print(len(labels), "фото | классы:", dict(enumerate(CLASSES)))
print("OpenCV", cv2.__version__, "| albumentations", A.__version__, "| imagehash", imagehash.__version__)

### Общие функции

Готовый код, менять не нужно.

- `read_xml(filename)` читает Pascal VOC XML фото: размер кадра из `<size>` и список объектов, у каждого `name`, флаги и рамка `bbox` = `[xmin, ymin, xmax, ymax]` в пикселях со счётом от 1. Если XML нет, возвращает `None`.
- `load_rgb(filename)` читает фото в исходном размере через `.convert("RGB")`. В этой лабе фото не уменьшаем: рамки и маски должны совпадать с исходными пикселями.
- `load_trimap(filename)` читает trimap: 1 = животное, 2 = фон, 3 = граница.
- `draw_boxes(rgb, boxes, color)` рисует рамки `[xmin, ymin, xmax, ymax]`, `show_grid` рисует картинки сеткой и сохраняет PNG.

In [ ]:
def read_xml(filename):
    """{"width", "height", "objects": [{"name", "truncated", "occluded", "difficult", "bbox"}]} или None."""
    path = XML_DIR / (Path(filename).stem + ".xml")
    if not path.exists():
        return None
    root = ET.parse(path).getroot()
    size = root.find("size")
    objects = []
    for obj in root.findall("object"):
        bb = obj.find("bndbox")
        objects.append({
            "name": obj.findtext("name"),
            "truncated": int(obj.findtext("truncated", "0")),
            "occluded": int(obj.findtext("occluded", "0")),
            "difficult": int(obj.findtext("difficult", "0")),
            "bbox": [int(float(bb.findtext(k))) for k in ("xmin", "ymin", "xmax", "ymax")],
        })
    return {"width": int(size.findtext("width")), "height": int(size.findtext("height")), "objects": objects}


def load_rgb(filename):
    with Image.open(IMG_DIR / filename) as im:
        return np.asarray(im.convert("RGB"))


def load_trimap(filename):
    with Image.open(TRIMAP_DIR / (Path(filename).stem + ".png")) as im:
        return np.asarray(im)


def draw_boxes(rgb, boxes, color=(255, 196, 0), thickness=None):
    out = np.ascontiguousarray(rgb.copy())
    th = thickness or max(2, round(max(rgb.shape[:2]) / 250))
    for b in boxes:
        x0, y0, x1, y1 = [int(round(v)) for v in b[:4]]
        cv2.rectangle(out, (x0, y0), (x1, y1), color, th)
    return out


def draw_mask_contour(rgb, mask, color=(46, 123, 255)):
    out = np.ascontiguousarray(rgb.copy())
    cs, _ = cv2.findContours(np.asarray(mask, np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(out, cs, -1, color, max(2, round(max(rgb.shape[:2]) / 300)))
    return out


def show_grid(images, titles=None, ncols=4, path=None, size=3.0):
    nrows = (len(images) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for i, img in enumerate(images):
        ax = axes.ravel()[i]
        img = np.asarray(img)
        ax.imshow(img, cmap="gray" if img.ndim == 2 else None)
        if titles is not None:
            ax.set_title(titles[i], fontsize=8)
    plt.tight_layout()
    if path is not None:
        plt.savefig(path, dpi=100)
    plt.show()


first = labels["filename"].iloc[0]
print(first, load_rgb(first).shape, read_xml(first))

# Часть 1. Разметка, форматы, дубликаты

Делается после лекции 4. Результат части: таблица проверки разметки, разметка в форматах COCO и YOLO с проверкой перевода, список пар-дубликатов с вашей отметкой и графики распределений. В часть 2 переходят отмеченные пары: из них соберутся группы для разбиения.

## 1.1 Проверка разметки

Допишите `audit_one`: словарь проверок для одного фото. Размер фото, наличие XML и число рамок готовая функция `base_info` уже считает. Готовая ячейка ниже пройдёт по всем фото, сохранит `results/audit.csv` и покажет фото, где хоть одна проверка не прошла.

| Поле | Что это |
|---|---|
| `format` | формат по первым байтам файла: `jpeg` (`FF D8`), `png` (`89 50 4E 47`), `gif` (`47 49 46`), иначе `other` |
| `mode` | режим Pillow: `RGB`, `L`, `RGBA`, `P` |
| `xml_size_ok` | размер в XML совпадает с фото (`True`, если XML нет) |
| `boxes_inside` | все рамки внутри кадра: `1 <= xmin < xmax <= width`, то же по y |
| `trimap_size_ok` | trimap того же размера, что фото |
| `trimap_empty` | в trimap нет ни одного пикселя 1 и 3 |
| `box_on_animal` | для первой рамки: доля её пикселей, где trimap не 2 (`NaN`, если рамки нет или маска пустая) |
| `box_area_frac` | площадь первой рамки / площадь кадра (`NaN`, если рамки нет) |

In [ ]:
def audit_one(filename):
    """Словарь проверок для одного фото, поля из таблицы выше."""
    # TODO: первые 8 байт: (IMG_DIR / filename).read_bytes()[:8]; формат по сигнатуре
    # TODO: Image.open(IMG_DIR / filename): .mode и .size (это (ширина, высота), нужен для проверок)
    # TODO: xml = read_xml(filename); рамки: [o["bbox"] for o in xml["objects"]]
    # TODO: tri = load_trimap(filename); пустая маска: not np.isin(tri, [1, 3]).any()
    # TODO: box_on_animal: вырез tri[ymin - 1:ymax, xmin - 1:xmax] (счёт в XML с 1), доля пикселей != 2
    return {"filename": filename}

In [ ]:
def base_info(filename):
    """Поля, которые нужны дальше в ноутбуке: размер, есть ли XML, сколько рамок."""
    with Image.open(IMG_DIR / filename) as im:
        w, h = im.size
    xml = read_xml(filename)
    return {"filename": filename, "width": w, "height": h, "has_xml": xml is not None,
            "n_boxes": len(xml["objects"]) if xml else 0}


# Пока audit_one не дописана, в таблице только base_info, остальные столбцы пустые.
CHECKS = ["format", "mode", "xml_size_ok", "boxes_inside", "trimap_size_ok", "trimap_empty", "box_on_animal", "box_area_frac"]
audit = pd.DataFrame([{**base_info(fn), **audit_one(fn)} for fn in labels["filename"]]).merge(labels, on="filename")
audit = audit.reindex(columns=list(dict.fromkeys(list(audit.columns) + CHECKS)))
audit.to_csv(RESULTS / "audit.csv", index=False, float_format="%.4f")
flag = lambda col, bad: audit[col].map(lambda v: v == bad if isinstance(v, (bool, np.bool_)) else False)
problems = audit[(audit["format"].notna() & (audit["format"] != "jpeg")) | (audit["mode"].notna() & (audit["mode"] != "RGB"))
                 | (audit["n_boxes"] > 1) | flag("xml_size_ok", False) | flag("boxes_inside", False)
                 | flag("trimap_size_ok", False) | flag("trimap_empty", True) | (audit["box_on_animal"] < 0.5)]
print(len(audit), "фото,", len(problems), "с замечаниями")
print(problems[["filename", "format", "mode", "n_boxes", "trimap_empty", "box_on_animal"]].to_string())

In [ ]:
# TODO: посмотрите на 3–4 фото из списка выше: фото, рамки draw_boxes и trimap (tri * 80).
#       Что из найденного — ошибка разметки, а что особенность датасета? Запишите в выводы.

## 1.2 COCO и YOLO

Разметка переводится в два формата, каждый под свою задачу курса:

- **COCO** для сегментации (лаба 6): объект — животное. Полигон `segmentation` получаем из маски trimap через `cv2.findContours`, `bbox` — рамка этой маски (`cv2.boundingRect`) в формате `[x, y, w, h]` со счётом от 0. Рамка головы сюда не идёт: в COCO рамка и маска должны описывать один объект.
- **YOLO** для детекции голов (лаба 5): по txt-файлу на фото с XML, строка `class cx cy w h` в долях кадра на каждую рамку головы.

Класс берите из `labels.csv`, а не из XML: в XML записан вид, `cat` или `dog`.

Сначала решите, что делать с границей trimap (значение 3): отнести к животному или к фону. Впишите решение в `BORDER_AS` и объясните его в отчёте.

In [ ]:
BORDER_AS = "animal"   # TODO: "animal" или "background" — решите сами и объясните в отчёте


def animal_mask(tri):
    """Бинарная маска животного uint8 {0, 1} с учётом BORDER_AS."""
    return ((tri == 1) | ((tri == 3) & (BORDER_AS == "animal"))).astype(np.uint8)


def voc_to_coco(b):
    """[xmin, ymin, xmax, ymax] со счётом от 1 → [x, y, w, h] со счётом от 0."""
    # TODO: x = xmin - 1, w = xmax - xmin + 1 и так же по y
    return [0, 0, 0, 0]


def coco_to_yolo(bbox, W, H):
    """[x, y, w, h] в пикселях → (cx, cy, w, h) в долях кадра."""
    # TODO: центр = угол + половина размера, всё делим на W или H
    return (0.0, 0.0, 0.0, 0.0)


def yolo_to_voc(cx, cy, w, h, W, H):
    """Обратный перевод для проверки: доли кадра → [xmin, ymin, xmax, ymax] со счётом от 1."""
    # TODO: пиксельные ширина и высота w * W, h * H; xmin = cx * W - w * W / 2 + 1; xmax = xmin + w * W - 1
    return [0, 0, 0, 0]


def mask_to_polygons(mask, eps=1.0):
    """Список полигонов COCO [[x1, y1, x2, y2, ...], ...] по внешним контурам маски."""
    # TODO: cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # TODO: упростите каждый контур cv2.approxPolyDP(c, eps, True), отбросьте контуры меньше 3 точек
    # TODO: c.reshape(-1).tolist() даёт плоский список x1, y1, x2, y2, ...
    return []


def polygons_to_mask(polygons, shape):
    """Обратно: заливка полигонов cv2.fillPoly в маску uint8 размером shape."""
    m = np.zeros(shape, np.uint8)
    pts = [np.array(p, np.int32).reshape(-1, 2) for p in polygons]
    if pts:
        cv2.fillPoly(m, pts, 1)
    return m

In [ ]:
# Готовая сборка COCO и YOLO из ваших функций.
coco = {"images": [], "annotations": [],
        "categories": [{"id": i, "name": name} for i, name in enumerate(CLASSES)]}
YOLO_DIR = RESULTS / "yolo"
(YOLO_DIR / "labels").mkdir(parents=True, exist_ok=True)
for old in (YOLO_DIR / "labels").glob("*.txt"):
    old.unlink()

for image_id, r in enumerate(audit.itertuples(), start=1):
    coco["images"].append({"id": image_id, "file_name": r.filename, "width": int(r.width), "height": int(r.height)})
    mask = animal_mask(load_trimap(r.filename))
    polys = mask_to_polygons(mask)
    if polys:
        x, y, w, h = cv2.boundingRect(polygons_to_mask(polys, mask.shape))
        coco["annotations"].append({"id": len(coco["annotations"]) + 1, "image_id": image_id,
                                    "category_id": int(r.class_id), "segmentation": polys,
                                    "bbox": [x, y, w, h], "area": int(mask.sum()), "iscrowd": 0})
    xml = read_xml(r.filename)
    if xml:
        lines = [f"{r.class_id} " + " ".join(f"{v:.6f}" for v in coco_to_yolo(voc_to_coco(o["bbox"]), r.width, r.height))
                 for o in xml["objects"]]
        (YOLO_DIR / "labels" / (Path(r.filename).stem + ".txt")).write_text("\n".join(lines) + "\n")

(RESULTS / "coco.json").write_text(json.dumps(coco))
(YOLO_DIR / "data.yaml").write_text(
    "# пути к фото задаются в лабе 5\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASSES)))
print("COCO:", len(coco["images"]), "фото,", len(coco["annotations"]), "объектов | YOLO:",
      len(list((YOLO_DIR / "labels").glob("*.txt"))), "файлов")

## 1.3 Проверка перевода

Две проверки, обе готовые. Первая читает рамки обратно из файлов YOLO, переводит их в VOC вашей `yolo_to_voc` и сравнивает с XML: расхождение должно быть меньше пикселя. Рамки из YOLO рисуются на 6 случайных фото красным поверх исходных зелёных. Вторая восстанавливает каждую маску из полигона COCO и считает IoU с исходной маской.

In [ ]:
diffs, panels, titles = [], [], []
with_xml = audit[audit["has_xml"]]
for r in with_xml.itertuples():
    xml = read_xml(r.filename)
    txt = (YOLO_DIR / "labels" / (Path(r.filename).stem + ".txt")).read_text().split("\n")
    for o, line in zip(xml["objects"], [t for t in txt if t]):
        cls, cx, cy, w, h = line.split()
        back = yolo_to_voc(float(cx), float(cy), float(w), float(h), r.width, r.height)
        diffs.append(max(abs(a - b) for a, b in zip(back, o["bbox"])))
for r in with_xml.sample(min(6, len(with_xml)), random_state=VARIANT).itertuples():
    xml = read_xml(r.filename); rgb = load_rgb(r.filename)
    txt = [t for t in (YOLO_DIR / "labels" / (Path(r.filename).stem + ".txt")).read_text().split("\n") if t]
    yolo_boxes = [yolo_to_voc(*map(float, t.split()[1:]), r.width, r.height) for t in txt]
    img = draw_boxes(draw_boxes(rgb, [o["bbox"] for o in xml["objects"]], (0, 212, 122)), yolo_boxes, (255, 59, 48))
    panels.append(img); titles.append(f"{r.filename} · {CLASSES[int(txt[0].split()[0])] if txt else '-'}")

ious = []
ann_by_image = {a["image_id"]: a for a in coco["annotations"]}
for img_info in coco["images"]:
    a = ann_by_image.get(img_info["id"])
    if a is None:
        continue
    m0 = animal_mask(load_trimap(img_info["file_name"]))
    m1 = polygons_to_mask(a["segmentation"], m0.shape)
    ious.append((m0 & m1).sum() / max((m0 | m1).sum(), 1))
print(f"рамки VOC → YOLO → VOC: максимальное расхождение {max(diffs) if diffs else float('nan'):.3f} px")
print(f"маска → полигон → маска: IoU медиана {np.median(ious) if ious else float('nan'):.4f}, минимум {min(ious) if ious else float('nan'):.4f}")
show_grid(panels, titles, ncols=3, path=RESULTS / "format_check.png", size=4)

## 1.4 Дубликаты

Ищем копии одного фото и серии снимков одного животного. Как в лекции 4: MD5 ловит побайтовые копии, перцептивные хэши упорядочивают остальные пары от похожих к непохожим, а решение «дубликат или нет» принимаете вы, глядя на картинки.

Допишите `image_hashes`: MD5 файла, `imagehash.phash` и `imagehash.dhash` фото, а также phash и dhash отражённого фото (`ImageOps.mirror`), чтобы ловить отражения. Готовые ячейки посчитают расстояния для всех пар, сохранят первые 30 пар в `results/dup_candidates.png` и подготовят словарь для ваших отметок.

In [ ]:
def image_hashes(filename):
    """{"md5": str, "phash": ImageHash, "dhash": ImageHash, "phash_m": ImageHash, "dhash_m": ImageHash}."""
    # TODO: md5 = hashlib.md5((IMG_DIR / filename).read_bytes()).hexdigest()
    # TODO: im = Image.open(IMG_DIR / filename).convert("RGB"); imagehash.phash(im), imagehash.dhash(im)
    # TODO: то же для ImageOps.mirror(im)
    return {}

In [ ]:
files = labels["filename"].tolist()
H = [image_hashes(fn) for fn in files]
N = len(files)
pairs = pd.DataFrame(columns=["a", "b", "dist", "exact"])
if H and H[0]:
    bits = lambda key: np.array([h[key].hash.ravel() for h in H])        # N × 64 бита
    P, D, Pm, Dm = bits("phash"), bits("dhash"), bits("phash_m"), bits("dhash_m")
    ham = lambda X, Y: (X[:, None, :] != Y[None, :, :]).sum(-1)          # попарное расстояние Хэмминга
    direct = ham(P, P) + ham(D, D)
    mirrored = ham(Pm, P) + ham(Dm, D)
    dist = np.minimum(direct, np.minimum(mirrored, mirrored.T))
    md5 = np.array([h["md5"] for h in H])
    i, j = np.triu_indices(N, k=1)
    pairs = pd.DataFrame({"a": np.array(files)[i], "b": np.array(files)[j], "dist": dist[i, j],
                          "mirror": (mirrored[i, j] < direct[i, j]) | (mirrored[j, i] < direct[i, j]),
                          "exact": md5[i] == md5[j]})
    pairs.loc[pairs["exact"], "dist"] = 0
    pairs = pairs.sort_values(["dist", "a"]).reset_index(drop=True)
print(N, "фото,", len(pairs), "пар | побайтовых копий:", int(pairs["exact"].sum()) if len(pairs) else 0)
pairs.head(10)

In [ ]:
TOP = pairs.head(30).copy()
TOP.insert(0, "rank", range(len(TOP)))
panels, titles = [], []
for r in TOP.itertuples():
    a, b = load_rgb(r.a), load_rgb(r.b)
    h = 200
    a = cv2.resize(a, (round(a.shape[1] * h / a.shape[0]), h)); b = cv2.resize(b, (round(b.shape[1] * h / b.shape[0]), h))
    panels.append(np.hstack([a, np.full((h, 8, 3), 255, np.uint8), b]))
    titles.append(f"#{r.rank} d={r.dist}{' md5' if r.exact else ''}{' mirror' if r.mirror else ''}\n{Path(r.a).stem} | {Path(r.b).stem}")
if panels:
    show_grid(panels, titles, ncols=5, path=RESULTS / "dup_candidates.png", size=3.4)
print("Скопируйте в следующую ячейку и отметьте каждую пару:")
print("VERDICT = {\n" + "".join(f'    {r.rank}: "{"same" if r.exact else ""}",   # {r.a} {r.b} d={r.dist}\n' for r in TOP.itertuples()) + "}")

In [ ]:
# TODO: для каждой пары из листа dup_candidates.png впишите одно из трёх:
#   "same"      — то же фото: копия, пересжатие, кроп, отражение, замена фона;
#   "series"    — другой снимок того же животного (та же сцена или то же животное в другом месте);
#   "different" — разные животные.
# Побайтовые копии (md5) уже отмечены "same". Ключ — номер пары # на картинке.
VERDICT = {}

In [ ]:
dup = TOP.copy()
dup["verdict"] = [VERDICT.get(k, "same" if ex else "") for k, ex in zip(dup["rank"], dup["exact"])]
# побайтовые копии за пределами первых 30 тоже попадают в список
extra = pairs[pairs["exact"]].iloc[len(TOP):] if len(pairs) else pairs
if len(extra):
    extra = extra.assign(rank=-1, verdict="same")
    dup = pd.concat([dup, extra[dup.columns]], ignore_index=True)
dup.to_csv(RESULTS / "dup_pairs.csv", index=False)
print(dup["verdict"].replace("", "не отмечено").value_counts().to_string())

## 1.5 Распределения

Один рисунок `results/distribution.png` из трёх графиков: число фото по породам, число фото с рамкой головы по породам, гистограмма `box_area_frac` (площадь рамки в долях кадра) с вертикальной линией медианы. Всё нужное есть в таблице `audit`.

In [ ]:
# TODO: fig, axes = plt.subplots(1, 3, figsize=(16, 4))
# TODO: audit["breed"].value_counts() и audit[audit["has_xml"]]["breed"].value_counts() столбцами
# TODO: audit["box_area_frac"].dropna() гистограммой (bins=30), axvline на медиане
# TODO: plt.savefig(RESULTS / "distribution.png", dpi=100)
# Напечатайте: медиану box_area_frac, долю рамок меньше 0.05 и минимальную сторону рамки в пикселях.

# Часть 2 (после лекции 5). Разбиение и аугментации

Из части 1 сюда приходит `VERDICT`: пары `same` и `series` становятся группами. Результат части: `split.csv`, на котором вы будете учить модели в лабах 4–6, и пайплайн аугментаций.

## 2.1 Группы

Фото — вершины графа, пары `same` и `series` — рёбра. Группа — связная компонента: все фото, до которых можно дойти по рёбрам. Фото без пар — группа из одного фото.

Допишите `make_groups`: верните `pd.Series` с номером группы для каждого фото из `files`.

In [ ]:
def make_groups(files, edges):
    """edges: список пар (filename_a, filename_b). Возвращает pd.Series filename -> номер группы."""
    # TODO: index = {fn: i for i, fn in enumerate(files)}
    # TODO: матрица смежности coo_matrix((np.ones(len(edges)), (rows, cols)), shape=(N, N))
    # TODO: n, comp = connected_components(adj, directed=False)
    return pd.Series(range(len(files)), index=files)   # пока каждое фото — своя группа

In [ ]:
edges = [(r.a, r.b) for r in dup.itertuples() if r.verdict in ("same", "series")]
group = make_groups(files, edges)
labels["group"] = labels["filename"].map(group)
sizes = labels["group"].value_counts()
multi = sizes[sizes > 1]
print(len(edges), "пар →", len(multi), "групп из 2+ фото, в них", int(multi.sum()), "фото")

## 2.2 Случайное разбиение

Готовая ячейка. 100 раз делим вариант случайно в пропорции 70/15/15 со стратификацией по породе и считаем, сколько групп оказалось разорвано между выборками и сколько групп попало одновременно в train и test. Это число утечек, которое дало бы разбиение без групп.

In [ ]:
def random_split(seed):
    tr, rest = train_test_split(labels, test_size=0.3, random_state=seed, stratify=labels["class_id"])
    va, te = train_test_split(rest, test_size=0.5, random_state=seed, stratify=rest["class_id"])
    return pd.concat([tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")])


def leaks(df):
    g = df[df["group"].isin(multi.index)].groupby("group")["split"]
    torn = int((g.nunique() > 1).sum())
    train_test = sum({"train", "test"} <= set(s) for _, s in g)
    return torn, train_test


rnd = pd.DataFrame([leaks(random_split(s)) for s in range(100)], columns=["torn", "train_test"])
print(f"случайное разбиение: разорвано групп в среднем {rnd['torn'].mean():.1f} из {len(multi)}, "
      f"одновременно в train и test {rnd['train_test'].mean():.1f}")

## 2.3 Групповое разбиение

Разделите вариант через `StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)`: первая часть в test, вторая в val, остальные в train. Получится примерно 70/15/15, группы целиком в одной выборке, доли пород близки. Впишите выборку в столбец `labels["split"]`. Готовая ячейка ниже проверит разбиение и сохранит `split.csv` и `split_summary.csv`.

In [ ]:
# TODO: sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
# TODO: folds = [test_idx for _, test_idx in sgkf.split(labels, labels["class_id"], labels["group"])]
# TODO: labels["split"] = "train"; labels.loc[folds[0], "split"] = "test"; labels.loc[folds[1], "split"] = "val"
labels["split"] = "train"

In [ ]:
torn, train_test = leaks(labels)
info = labels.merge(audit[["filename", "has_xml", "trimap_empty"]], on="filename")
by_split = pd.DataFrame({
    "photos": info.groupby("split").size(),
    "with_box": info[info["has_xml"]].groupby("split").size(),
    "with_mask": info[info["trimap_empty"] != True].groupby("split").size(),
}).reindex(["train", "val", "test"])
shares = pd.crosstab(info["breed"], info["split"], normalize="columns").reindex(columns=["train", "val", "test"]).round(3)
print("разорванных групп:", torn, "| групп в train и test:", train_test)
print(by_split.to_string()); print(shares.to_string())

labels[["filename", "class_id", "group", "split"]].to_csv(RESULTS / "split.csv", index=False)
summary = {"variant": VARIANT, "seed": SEED, "groups_multi": len(multi), "photos_in_groups": int(multi.sum()),
           "random_torn_mean": round(rnd["torn"].mean(), 2), "random_train_test_mean": round(rnd["train_test"].mean(), 2),
           "group_split_torn": torn,
           **{f"{s}_{c}": int(by_split.loc[s, c]) for s in by_split.index for c in by_split.columns if pd.notna(by_split.loc[s, c])},
           "breed_share_min": float(shares.values.min()), "breed_share_max": float(shares.values.max())}
pd.Series(summary).to_csv(RESULTS / "split_summary.csv", header=["value"], index_label="metric")
pd.Series(summary)

## 2.4 Аугментации

Соберите два пайплайна albumentations:

- `train_transform`: отражение, умеренный масштаб и сдвиг (`A.Affine`), яркость и контраст, плюс минимум одна аугментация на ваш выбор. Каждая вероятностная, `p < 1`;
- `eval_transform` для val и test: только изменение размера (`A.LongestMaxSize(max_size=640)`).

Оба получают фото, маску животного и рамки голов в формате `pascal_voc` и возвращают их преобразованными. Параметр `bbox_params` уже задан. Готовая ячейка нарисует сетку 4×4: 4 фото из train по 4 варианта, рамка жёлтая, контур маски синий. Посмотрите на неё как на слайде «Аудит» лекции 5.

In [ ]:
BBOX = A.BboxParams(format="pascal_voc", label_fields=["labels"], min_visibility=0.0)

# TODO: train_transform = A.Compose([A.HorizontalFlip(p=0.5), A.Affine(...), A.RandomBrightnessContrast(...), ...],
#                                   bbox_params=BBOX, seed=SEED)
train_transform = A.Compose([A.NoOp()], bbox_params=BBOX, seed=SEED)
# TODO: eval_transform = A.Compose([A.LongestMaxSize(max_size=640)], bbox_params=BBOX)
eval_transform = A.Compose([A.NoOp()], bbox_params=BBOX)

In [ ]:
def apply(tf, filename):
    rgb, tri, xml = load_rgb(filename), load_trimap(filename), read_xml(filename)
    boxes = [o["bbox"] for o in xml["objects"]] if xml else []
    r = tf(image=rgb, mask=animal_mask(tri), bboxes=boxes, labels=[0] * len(boxes))
    return r["image"], r["mask"], r["bboxes"]


demo = labels[(labels["split"] == "train") & labels["filename"].isin(audit.loc[audit["has_xml"] & (audit["trimap_empty"] != True), "filename"])]
demo = demo.groupby("class_id").head(1)["filename"].tolist()[:4]
panels, titles = [], []
for fn in demo:
    for k in range(4):
        img, m, bb = apply(train_transform, fn)
        panels.append(draw_boxes(draw_mask_contour(img, m), bb)); titles.append(f"{Path(fn).stem} #{k + 1}")
show_grid(panels, titles, ncols=4, path=RESULTS / "aug_audit.png", size=3.2)

## 2.5 Аугментация, которая ломает

Выберите одно преобразование или настройку, которая портит разметку или меняет правильный ответ: сильный кроп, поворот на большой угол, сдвиг оттенка для вашей трудной пары пород или своё. Впишите его в `BAD_TRANSFORM`: готовая ячейка нарисует 4 примера в `results/aug_bad.png`.

Затем посчитайте на всех фото с рамкой, в какой доле случаев разметка испорчена. Для геометрии есть две готовые функции:

- `crop_visible(box, W, H, rng, scale, ratio)` — доля головы, которая попала в случайный кроп с параметрами `RandomResizedCrop`;
- `rotated_box_iou(box, angle)` — IoU рамки, описанной вокруг повёрнутой рамки, с самой рамкой.

Критерий выберите сами и запишите его в выводы: голова целиком вне кадра, видно меньше половины головы, IoU меньше 0,7 или свой.

In [ ]:
def crop_visible(box, W, H, rng, scale=(0.08, 1.0), ratio=(3 / 4, 4 / 3)):
    """Доля площади рамки [xmin, ymin, xmax, ymax], попавшая в случайный кроп как у RandomResizedCrop."""
    for _ in range(10):
        area = W * H * rng.uniform(*scale)
        ar = np.exp(rng.uniform(np.log(ratio[0]), np.log(ratio[1])))
        w, h = int(round(np.sqrt(area * ar))), int(round(np.sqrt(area / ar)))
        if 0 < w <= W and 0 < h <= H:
            x0, y0 = rng.integers(0, W - w + 1), rng.integers(0, H - h + 1)
            break
    else:
        x0, y0, w, h = 0, 0, W, H
    ix = max(0, min(box[2], x0 + w) - max(box[0] - 1, x0))
    iy = max(0, min(box[3], y0 + h) - max(box[1] - 1, y0))
    return ix * iy / ((box[2] - box[0] + 1) * (box[3] - box[1] + 1))


def rotated_box_iou(box, angle):
    """IoU тесной рамки и рамки вдоль осей вокруг неё же, повёрнутой на angle градусов."""
    w, h = box[2] - box[0] + 1, box[3] - box[1] + 1
    a = np.deg2rad(angle)
    W2 = abs(w * np.cos(a)) + abs(h * np.sin(a)); H2 = abs(w * np.sin(a)) + abs(h * np.cos(a))
    return (w * h) / (W2 * H2)

In [ ]:
# TODO: BAD_TRANSFORM = A.Compose([...], bbox_params=BBOX, seed=SEED), например
#       A.RandomResizedCrop(size=(320, 320), scale=(0.08, 1.0)) или A.Rotate(limit=(30, 30), p=1)
BAD_TRANSFORM = A.Compose([A.NoOp()], bbox_params=BBOX, seed=SEED)
BAD_NAME = "TODO: название и параметры"

panels = []
for fn in demo:
    img, m, bb = apply(BAD_TRANSFORM, fn)
    panels.append(draw_boxes(draw_mask_contour(img, m), bb))
show_grid(panels, [BAD_NAME] * len(panels), ncols=4, path=RESULTS / "aug_bad.png", size=3.4)

# TODO: доля испорченных случаев на всех фото с рамкой, например для кропа:
#       rng = np.random.default_rng(SEED)
#       vis = [crop_visible(b, w, h, rng) for ... for _ in range(20)]
#       print("голова вне кадра:", np.mean(np.array(vis) == 0), "| меньше половины:", np.mean(np.array(vis) < 0.5))

## Выгрузка

Ячейка ниже проверяет, что все 11 результатов на месте, и печатает главное для отчёта. Затем упакуйте `results/` в архив и скачайте:
- в Kaggle: правая панель → Output → `results.zip` → Download; ноутбук: File → Download Notebook;
- в Colab: архив скачается сам после запуска ячейки; ноутбук: File → Download → Download .ipynb;
- локально: ничего скачивать не нужно, всё уже лежит в папке `lab03/`.

Переименуйте скачанный ноутбук в `solution.ipynb`.

In [ ]:
print(f"вариант {VARIANT} | BORDER_AS = {BORDER_AS} | отмечено пар: {sum(v != '' for v in dup['verdict'])} | "
      f"групп из 2+ фото: {len(multi)} | split: {labels['split'].value_counts().to_dict()}")
expected = {"audit.csv", "coco.json", "yolo", "format_check.png", "dup_pairs.csv", "dup_candidates.png",
            "distribution.png", "split.csv", "split_summary.csv", "aug_audit.png", "aug_bad.png"}
missing = sorted(expected - {p.name for p in RESULTS.iterdir()})
print("не хватает:", ", ".join(missing) if missing else "ничего, все 11 результатов на месте")

In [ ]:
import shutil
if ENV != "local":
    archive = shutil.make_archive("results", "zip", RESULTS)
    print("архив:", archive)
    if ENV == "colab":
        from google.colab import files
        files.download(archive)